# CHIIR 2026 Tutorial on Controlled Experimentation of Model Search Behaviour with Geniie-Lab

## Indexing (SPLADE)

### Install python modules

In [1]:
import sys
!{sys.executable} -m pip install ir_datasets pandas opensearch-py nltk sentence_transformers torch==2.9.1 torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130


### Load helper modules

In [2]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

In [3]:
from opensearchpy import OpenSearch

In [4]:
host = 'localhost'
port = 9200

client = OpenSearch(
    hosts = [{'host': host, 'port': port}],
    http_compress = True,
    use_ssl = False,
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)

In [5]:
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'eGHrQd-_TRCFMeuGXEHjLg',
 'name': 'd7c391a16c06',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2025-10-29T22:22:22.753988939Z',
             'build_hash': '6564992150e26aaa62d4522a220dfff5188aeb88',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.1',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.3.2'}}


### Index a Corpus for SPLADE Model

- Note: You should have a GPU for indexing.
- We're using an "inference-free" model which does not expand queries.
    - See [https://sbert.net/docs/sparse_encoder/pretrained_models.html](https://sbert.net/docs/sparse_encoder/pretrained_models.html)

In [6]:
import ir_datasets
dataset_name = "beir/scidocs"
dataset = ir_datasets.load(dataset_name)

In [7]:
print(dataset.docs_cls().__annotations__)

{'doc_id': <class 'str'>, 'text': <class 'str'>, 'title': <class 'str'>, 'authors': typing.List[str], 'year': <class 'int'>, 'cited_by': typing.List[str], 'references': typing.List[str]}


Index structure

In [8]:
index_name = "scidocs_splade"
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

In [9]:
index_body ={
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
  },
  "mappings": {
    "properties": {
      "docid": { "type": "keyword" },
      "title": { "type": "text" },
      "text": { "type": "text" },
      "sparse_embedding": {
        "type": "rank_features"
      }
    }
  }
}
response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

{'acknowledged': True, 'index': 'scidocs_splade', 'shards_acknowledged': True}


Encoding Model

In [10]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [11]:
from sentence_transformers.sparse_encoder import SparseEncoder
# encoder_model = "naver/splade-cocondenser-ensembledistil"
encoder_model = "opensearch-project/opensearch-neural-sparse-encoding-doc-v3-distill"
model = SparseEncoder(encoder_model, trust_remote_code=True).to(device)

/home/hideo/GitHub/chiir2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1528.10it/s, Materializing param=vocab_transform.weight]                                
/home/hideo/GitHub/chiir2026/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


Indexing

In [12]:
for doc in tqdm(dataset.docs_iter(), desc="Indexing"):
    doc_body = {
        "docid": doc.doc_id,
        "title": doc.title,
        "text":  doc.text,
    }
    doc_tensor = model.encode_document([f"{doc.title}\n{doc.text}"])
    doc_embedding = model.decode(doc_tensor)
    doc_body["sparse_embedding"] = dict(doc_embedding[0])
    response = client.index(index=index_name, body=doc_body)

Indexing: 25657it [05:44, 74.55it/s]


#### Search Test

In [13]:
def search(query: str, size: int = 10) -> dict:
    query_tensor = model.encode_query([query])
    query_embedding = model.decode(query_tensor)
    query_body = {
        "size": size,
        "query": {
            "neural_sparse": {
                "sparse_embedding": {
                    "query_tokens": dict(query_embedding[0])
                }
            }
        }
    }
    return client.search(index=index_name, body=query_body)

In [14]:
q = "Ad Hoc Retrieval Experiments Using WordNet"
resp = search(q, size=5)

print(f"\nTop {len(resp['hits']['hits'])} hits for query: {q}\n")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")


Top 5 hits for query: Ad Hoc Retrieval Experiments Using WordNet

[59407446503d49a8cf5f5643b17502835b62f139] Using WordNet to Disambiguate Word Senses for Text... (score=13.89)
[62eff7763f8679d0afe53dad4d85279d54f763c5] Using WordNet as a Knowledge Base for Measuring Se... (score=12.75)
[1cc7013247056e45264de9817171d72690181692] A language modeling framework for resource selecti... (score=12.43)
[c43826e860dfd9365aa8905397393d96513d1daa] Tapping into knowledge base for concept feedback: ... (score=11.09)
[8b40b159c2316dbea297a301a9c561b1d9873c4a] Monolingual and Cross-Lingual Information Retrieva... (score=10.85)
